In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/gk/dmjzqj790nv950czzbz85w6r0000gn/T/ipykernel_40078/4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/Users/roshni/Documents/Hasib/Personal Project/YTRAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory

def process_all_pdf(pdf_directory):
    """Process all the pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf diles recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n processing :{pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source infor to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            print(f"all documetn now-> {len(all_documents)}")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\n Total Documetns loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf("../data")

Found 2 PDF files to process

 processing :gs statement uwa.pdf
Loaded 2 pages
all documetn now-> 2

 processing :Sanjana_SOP_UWA_MIT_v4.pdf
Loaded 7 pages
all documetn now-> 9

 Total Documetns loaded: 9


In [3]:
output_file = "../data/text_files/combined_documents.txt"

with open(output_file,"w",encoding="utf-8") as f:
    f.write(all_pdf_documents[0].page_content)
    f.write(all_pdf_documents[1].page_content)
    

In [4]:
### Text splitting

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks")

    #show example of chunk
    if split_doc:
        print(f"\n Example chunk:")
        print(f"content: {split_doc[0].page_content[:200]}...")
        print(f"metadata: {split_doc[0].metadata}")

    return split_doc

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 9 documents into 27 chunks

 Example chunk:
content: 1. I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is 
employed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a 
real estate company specializing in r...
metadata: {'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': '', 'creationdate': "D:20260711172409Z00'00'", 'source': '../data/pdf/gs statement uwa.pdf', 'file_path': '../data/pdf/gs statement uwa.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260711172409Z00'00'", 'trapped': '', 'modDate': "D:20260711172409Z00'00'", 'creationDate': "D:20260711172409Z00'00'", 'page': 0, 'source_file': 'gs statement uwa.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': '', 'creationdate': "D:20260711172409Z00'00'", 'source': '../data/pdf/gs statement uwa.pdf', 'file_path': '../data/pdf/gs statement uwa.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260711172409Z00'00'", 'trapped': '', 'modDate': "D:20260711172409Z00'00'", 'creationDate': "D:20260711172409Z00'00'", 'page': 0, 'source_file': 'gs statement uwa.pdf', 'file_type': 'pdf'}, page_content='1. I, Sanjana Akter Roshni, reside in Dhaka, Bangladesh, with my husband, who is \nemployed as a Software Engineer. My father owns and manages Sydney Homes Ltd, a \nreal estate company specializing in residential apartment construction and sales. I am not \ncurrently involved in any community or NGO leadership roles. \nMy primary sponsor is my father, with fixed deposit of BDT 50,00,000 (approximately \nAUD 57,145). My secondary sponsor is

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name: str = "BAAI/bge-small-en-v1.5"):
        """ 
        Initialize the embedding manager

        Args: model_name: HuggingFace model name for sentense embeddings 
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """ 
        Generate embedding for a list of texts

        Args: 
            texts: List of text strings to embed
        Returns:
            numpy array of embedding with shape (len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddinf for {len(texts)} texts ...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12825.24it/s]


Model loaded successfully. Embedding dimension: 384


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description" : "pdf document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initialize vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any],embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection : 0


In [9]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddinf for 27 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.38it/s]

Generated embeddings with shape (27, 384)
Adding 27 documents to vector store...
Successfully added 27 documents to vector store
Total documents in collection: 27


In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [11]:
context = rag_retriever.retrieve("Nasima Akter")
for c in context:
    print(c.get('content'))

Retrieving documents for query: 'Nasima Akter'
Top K: 5, Score threshold: 0.0
Generating embeddinf for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.16it/s]

Generated embeddings with shape (1, 384)
Retrieved 0 documents (after filtering)


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer=rag_simple("Tell me in details about sanjana course?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Tell me about bKash and SMS gateways'
Top K: 3, Score threshold: 0.0
Generating embeddinf for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.91it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


bKash and SMS gateways are third-party APIs that I have integrated into our platforms at Ease Limited in Dhaka. 

bKash is a popular mobile financial service in Bangladesh that allows users to send and receive money, pay bills, and make purchases using their mobile phones. I have integrated bKash into our platforms to enable payment processing services, allowing users to make transactions using their bKash accounts.

SMS gateways, on the other hand, are services that enable the sending and receiving of SMS messages programmatically. I have integrated SMS gateways into our platforms to send and receive transactional messages, such as payment confirmations and notifications, to users. This integration has helped to enhance the user experience and improve the overall efficiency of our transactional systems.


In [44]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Tell me in details about sanjana husband", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Tell me in details about sanjana husband'
Top K: 3, Score threshold: 0.1
Generating embeddinf for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.63it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


Answer: Sanjana's husband is Hasibul Hoque, whose passport number is A03116434. He is a Software Engineer with over three years of experience specializing in Java backend systems, microservices, and enterprise banking software. He has a thriving career in Bangladesh and is an established professional in his field.

Hasibul is joining Sanjana in Australia as her dependent, purely because of her studies. He will be providing emotional and practical support to Sanjana as she navigates an unfamiliar environment and focuses on her demanding master's program. Once Sanjana completes her degree, they intend to return home to Bangladesh together to resume their lives and careers.

Hasibul also has a financial presence in Bangladesh, with Mudaraba Term Deposits of approximately AUD 5,833. This indicates that their financial foundations are established in Bangladesh, rather than abroad.
Sources: [{'source': 'gs statement uwa.pdf', 'page': 0, 'score': 0.32119321823120117, 'preview': '1. I, Sanjana

In [52]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Tell me in details about sanjana", top_k=3, min_score=0.2, summarize=False)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Tell me in details about sanjana'
Top K: 3, Score threshold: 0.2
Generating embeddinf for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.17it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)



Final Answer: Sanjana Akter Roshni is a Bangladeshi resident of Dhaka, Bangladesh. She is married to a software engineer and her father owns and manages a real estate company called Sydney Homes Ltd. 

Sanjana has been working as a programmer at Computer Ease Limited since May 12, 2024, earning a monthly income of BDT 41,600. She has no study gaps and no military obligations. 

Sanjana's primary sponsor is her father, Md. Shahadat Hossain Shaheen, who owns a fixed deposit of BDT 50,00,000 (approximately AUD 56,186) and has an annual business income of BDT 35,50,000 (AUD 39,892). Her secondary sponsor is her mother-in-law, Mrs. Nasima Akter, who has a Mudaraba Term Deposit of BDT 40,00,000 (approximately AUD 44,949) and earns around AUD 15,507 a year in rental income.

Sanjana has a clear reason for pursuing a Master of Information Technology degree at the University of Western Australia (UWA), which she believes is the right next step for her career. She has strong personal ties to Ba